## Model Selection & Heart Disease Risk Scoring

In this notebook, we load the best-performing trained machine learning model from MLflow and use it to generate heart disease risk predictions.

The goal is to convert model outputs into clear, actionable risk levels (Low / Medium / High) that can be easily interpreted in a healthcare setting.

###Import Required Libraries

In [0]:
from pyspark.sql.functions import col, when
import mlflow
import mlflow.spark

### Step 2: Load ML-Ready Feature Table

In [0]:
features_df = spark.table("workspace.default.gold_heart_features")\
    .select("patient_id", "features", 'heart_disease_flag')
features_df.show(5)


### Step 3: Identify & Load the Champion Model

Based on MLflow evaluation, Random Forest achieved the highest AUC and is selected
as the champion model for inference.

In [0]:
model_uri = "models:/Heart_Disease_RF_Model@production"
model = mlflow.spark.load_model(model_uri, dfs_tmpdir="/Volumes/workspace/default/mlflow_tmp/")

### Step 4: Generate Predictions

In [0]:
predictions = model.transform(features_df)
predictions.select("patient_id","probability", "prediction").show(5)


### Step 5: Extract Risk Probability

In [0]:
from pyspark.sql.functions import col, udf
from pyspark.ml.linalg import Vector

# Minimal UDF to extract probability value
@udf('double')
def get_prob(v):
    if isinstance(v, Vector):
        return float(v[1])
    return None

predictions = predictions.withColumn(
    "risk_probability",
    get_prob(col("probability"))
)

### Step 6: Map Probability → Risk Level

In [0]:
predictions = predictions.withColumn(
    "risk_level",
    when(col("risk_probability") >= 0.7, "High Risk")
    .when(col("risk_probability") >= 0.4, "Medium Risk")
    .otherwise("Low Risk")
)


### Step 7: Select Final Columns for Analytics

In [0]:
final_predictions = predictions.select(
    "patient_id",
    "risk_probability",
    "risk_level",
    "heart_disease_flag"
)
final_predictions.show(10)


### Step 8: Save Final Predictions

In [0]:
final_predictions.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .saveAsTable("workspace.default.gold_heart_risk_predictions")


### Outcome

- Selected champion Random Forest model
- Generated patient-level risk probabilities
- Classified patients into Low / Medium / High risk
- Persisted results as a governed Gold Delta table for analytics
